# NB29 The quantum transition — Chapter 29

A chapter notebook for *Blockchain for Lawyers: A Reading Through Technology and Economics*. The code is the same as in the text.

- All code here is a **toy implementation** meant to make a principle visible. Do not use it with real assets, real private keys or real transactions.
- No external API is called, and everything runs without a network. Cells that need the bundled snapshot data carry that data inside the cell.
- Standard library only (run and checked on Python 3.13). There is no install cell.
- The outputs are the editor's own run, stored as produced. Running the cells again should give the same values (random seeds are fixed).

In [ ]:
# NB29-1: ECDLP resources for Shor's algorithm — n=256 plugged into the general formula of Roetteler et al. (2017), and the logical resources of four papers compared (for study)
import math

def roetteler_bound(n):
    """Upper-bound formula in the abstract of Roetteler et al. (2017): logical qubits and Toffoli gates."""
    qubits = 9 * n + 2 * math.ceil(math.log2(n)) + 10
    toffoli = 448 * n**3 * math.log2(n) + 4090 * n**3
    return qubits, toffoli

for n in (192, 224, 256, 384, 521):
    q, t = roetteler_bound(n)
    print(f"n={n:>3}-bit: logical qubit bound {q:>5},  Toffoli bound {t:.3e}")

q256, t256 = roetteler_bound(256)
print()
print("n=256 — Toffoli bound from the formula", f"{t256:.4e}", "| value the paper reports from simulation: about 1.26e11 (the formula is an upper bound, so slightly larger)")
print("The 1.28e11 that Aggarwal et al. (2018) plugged in is the formula's bound as is →", f"{t256/1.28e11:.3f}", "x")

papers = [  # (year, paper, logical qubits, Toffoli or T gate count)
    (2017, "Roetteler et al. (P-256)",           2330, 1.26e11),
    (2018, "Aggarwal et al. (incl. 4 ancilla)",  2334, 1.28e11),
    (2022, "Webber et al. circuit (T gates)",    2871, 5.76e9),
    (2026, "Babbush et al. — low-qubit circuit", 1200, 9.0e7),
    (2026, "Babbush et al. — low-gate circuit",  1450, 7.0e7),
]
print()
print(f"{'Year':<6}{'Paper':<34}{'Logical qubits':>15}{'Gates':>12}{'Gates vs 2017':>20}")
for y, name, lq, gates in papers:
    ratio = "different unit (T)" if "T gates" in name else f"{1.26e11/gates:.0f}x"
    print(f"{y:<6}{name:<34}{lq:>15,}{gates:>12.2e}{ratio:>20}")
print()
print("Physical qubits are not computed here: they are a function of error rate, code cycle, reaction time, connectivity and target time (Section 2, advanced box).")

n=192-bit: logical qubit bound  1754,  Toffoli bound 5.300e+10
n=224-bit: logical qubit bound  2042,  Toffoli bound 8.528e+10
n=256-bit: logical qubit bound  2330,  Toffoli bound 1.287e+11
n=384-bit: logical qubit bound  3484,  Toffoli bound 4.494e+11
n=521-bit: logical qubit bound  4719,  Toffoli bound 1.150e+12

n=256 — Toffoli bound from the formula 1.2875e+11 | value the paper reports from simulation: about 1.26e11 (the formula is an upper bound, so slightly larger)
The 1.28e11 that Aggarwal et al. (2018) plugged in is the formula's bound as is → 1.006 x

Year  Paper                              Logical qubits       Gates       Gates vs 2017
2017  Roetteler et al. (P-256)                    2,330    1.26e+11                  1x
2018  Aggarwal et al. (incl. 4 ancilla)           2,334    1.28e+11                  1x
2022  Webber et al. circuit (T gates)             2,871    5.76e+09  different unit (T)
2026  Babbush et al. — low-qubit circuit          1,200    9.00e+07               

In [ ]:
# NB29-2: exposed amounts by type in the reference-date snapshot (a secondary tally) — reads the bundled file and recomputes subtotals and shares (for study)
import json, os
from decimal import Decimal

FALLBACK = {  # the same values as data/exposed_output_types_2026-10-25.json (for environments without the file)
    "height": 972132, "created": "2026-10-19T10:28:55Z", "issued": "20100415.625",
    "rows": [("p2pk_script_type", "1715076.99695260"), ("taproot_script_type", "217063.18329201"),
             ("multisig_script_type", "416.60645555"), ("pubkeyhash_reuse", "1185672.15392868"),
             ("witness_v0_keyhash_reuse", "2098570.39816002"), ("scripthash_reuse", "1199814.71613438"),
             ("witness_v0_scripthash_reuse", "705381.38634311"), ("witness_unknown_reuse", "0.00000000")]}

try:
    with open(os.path.join("data", "exposed_output_types_2026-10-25.json"), encoding="utf-8") as f:
        snap = json.load(f)
    prim = snap["primary"]
    height, created = prim["snapshot_block_height"], prim["snapshot_created_at_utc"][:19]
    issued = Decimal(snap["denominator"]["issued_supply_btc"])
    rows = [(r["category"], r["btc"]) for r in prim["by_category"]]
    src = "bundled file"
except FileNotFoundError:
    height, created, issued = FALLBACK["height"], FALLBACK["created"], Decimal(FALLBACK["issued"])
    rows, src = FALLBACK["rows"], "constants (no file)"

print(f"Source: {src} | snapshot block {height:,} ({created}) | 2nd-hand tally (3rd-party SQL, not rerun) | denominator = scheduled issuance {issued:,} BTC (upper bound)")
print(f"{'Category':<30}{'BTC':>18}{'of issuance':>12}")
script_type = Decimal(0); reuse = Decimal(0)
for cat, btc in rows:
    v = Decimal(btc)
    if cat.endswith("script_type"): script_type += v
    else: reuse += v
    print(f"{cat:<30}{v:>18,.3f}{v / issued * 100:>11.3f}%")
total = script_type + reuse
print("-" * 60)
print(f"{'Subtotal: key in script':<30}{script_type:>18,.3f}{script_type / issued * 100:>11.3f}%")
print(f"{'Subtotal: revealed by reuse':<30}{reuse:>18,.3f}{reuse / issued * 100:>11.3f}%")
print(f"{'Total':<30}{total:>18,.3f}{total / issued * 100:>11.3f}%")
print()
print("How to read: the denominator is an upper bound on the circulating supply, so the shares lean low. The reuse categories also count scripts without keys as exposed, so they lean high.")
print("Off-chain exposure (xpubs, signed messages) and exposure while in the mempool are not counted. No conversion to prices is made.")

Source: bundled file | snapshot block 972,132 (2026-10-19T10:28:55) | 2nd-hand tally (3rd-party SQL, not rerun) | denominator = scheduled issuance 20,100,415.625 BTC (upper bound)
Category                                     BTC of issuance
p2pk_script_type                   1,715,076.997      8.533%
taproot_script_type                  217,063.183      1.080%
multisig_script_type                     416.606      0.002%
pubkeyhash_reuse                   1,185,672.154      5.899%
witness_v0_keyhash_reuse           2,098,570.398     10.440%
scripthash_reuse                   1,199,814.716      5.969%
witness_v0_scripthash_reuse          705,381.386      3.509%
witness_unknown_reuse                      0.000      0.000%
------------------------------------------------------------
Subtotal: key in script            1,932,556.787      9.615%
Subtotal: revealed by reuse        5,189,438.655     25.818%
Total                              7,121,995.441     35.432%

How to read: the denominat

In [ ]:
# NB29-3: tells the output type, and "whether the public key is exposed at the moment of receipt," from the head of the locking script (for study — fake bytes, not for real use)
GENESIS_SPK = bytes.fromhex(  # scriptPubKey of the genesis coinbase output in Chapter 12 (67 bytes): PUSH65 <public key> OP_CHECKSIG
    "4104678afdb0fe5548271967f1a67130b7105cd6a828e03909a67962e0ea1f61deb649f6bc3f4cef38c4f35504e51ec112de5c384df7ba0b8d578a4c702b6bf11d5fac")

def classify(spk: bytes):
    """Standard locking-script type → (type, key exposed on receipt?, note). Same judgment as the 'long exposure' column of the BIP-360 table."""
    n = len(spk)
    if n in (67, 35) and spk[0] in (0x41, 0x21) and spk[-1] == 0xAC:
        return "P2PK", True, "public key (65 or 33 bytes) sits in the script as is"
    if n == 25 and spk[:3] == b"\x76\xa9\x14" and spk[-2:] == b"\x88\xac":
        return "P2PKH", False, "only the HASH160 of the public key (20 bytes) — key revealed when spent, vulnerable if reused"
    if n == 23 and spk[:2] == b"\xa9\x14" and spk[-1] == 0x87:
        return "P2SH", False, "only the HASH160 of the script — script revealed when spent"
    if n == 22 and spk[:2] == b"\x00\x14":
        return "P2WPKH", False, "SegWit v0, key hash of 20 bytes"
    if n == 34 and spk[:2] == b"\x00\x20":
        return "P2WSH", False, "SegWit v0, script hash of 32 bytes"
    if n == 34 and spk[:2] == b"\x51\x20":
        return "P2TR", True, "SegWit v1 (Taproot), tweaked x-only public key of 32 bytes sits there as is"
    if n == 34 and spk[:2] == b"\x52\x20":
        return "P2MR (proposed, BIP-360 Draft)", False, "proposed SegWit v2 — only the Merkle root of the script tree, 32 bytes, no key path"
    if n >= 3 and 0x51 <= spk[0] <= 0x60 and spk[-1] == 0xAE:
        return "P2MS (bare multisig)", True, "OP_m <keys> OP_n OP_CHECKMULTISIG — the public keys sit there as is"
    return "nonstandard/unclassified", None, "this classifier knows only the standard types"

samples = [
    ("genesis output (real bytes)", GENESIS_SPK),
    ("P2PKH shape (fake)",  b"\x76\xa9\x14" + bytes(20) + b"\x88\xac"),
    ("P2SH shape (fake)",   b"\xa9\x14" + bytes(20) + b"\x87"),
    ("P2WPKH shape (fake)", b"\x00\x14" + bytes(20)),
    ("P2WSH shape (fake)",  b"\x00\x20" + bytes(32)),
    ("P2TR shape (fake)",   b"\x51\x20" + bytes(32)),
    ("P2MR shape (fake)",   b"\x52\x20" + bytes(32)),
    ("P2MS 1-of-2 shape (fake)", b"\x51" + b"\x21" + bytes(33) + b"\x21" + bytes(33) + b"\x52\xae"),
]
print(f"{'Sample':<28}{'Bytes':>6}  {'Type':<32}{'Key exposed on receipt':<24}Note")
for name, spk in samples:
    kind, exposed, why = classify(spk)
    flag = {True: "yes", False: "no*", None: "?"}[exposed]
    print(f"{name:<28}{len(spk):>6}  {kind:<32}{flag:<24}{why}")
print()
print("* no = safe as long as it only receives. Spending reveals the key or script, so reusing the same address makes it vulnerable (the No* note in the BIP-360 table).")
print("Leading byte of the public key in the genesis output:", GENESIS_SPK[1:2].hex(), "(04 = uncompressed, 65 bytes) — written on the chain as is since 2009.")

Sample                       Bytes  Type                            Key exposed on receipt  Note
genesis output (real bytes)     67  P2PK                            yes                     public key (65 or 33 bytes) sits in the script as is
P2PKH shape (fake)              25  P2PKH                           no*                     only the HASH160 of the public key (20 bytes) — key revealed when spent, vulnerable if reused
P2SH shape (fake)               23  P2SH                            no*                     only the HASH160 of the script — script revealed when spent
P2WPKH shape (fake)             22  P2WPKH                          no*                     SegWit v0, key hash of 20 bytes
P2WSH shape (fake)              34  P2WSH                           no*                     SegWit v0, script hash of 32 bytes
P2TR shape (fake)               34  P2TR                            yes                     SegWit v1 (Taproot), tweaked x-only public key of 32 bytes sits there as is
P